# 第12回　不均衡データと ROC
***
> **前提**: 第6回の評価指標を発展させ，クラス不均衡への対処と ROC/PR 曲線を学びます。第8回問題4で触れた Precision / Recall / F1 も本回で実践します。

> ⚠️ **この課題で身につけること：コーディングではなく「AI（機械学習）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや閾値を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります。AI に頼り切らず、要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

## 目次
1. 不均衡データの生成
2. class_weight
3. ROC 曲線と AUC
4. 適合率-再現率曲線

---

## この回で学ぶこと

### クラス不均衡が問題になる場面

現実の分類問題では，クラスの数が極端に偏っていることが多い：

| 問題 | クラス比率 |
|---|---|
| クレジットカード不正検知 | 正常 99.8% / 不正 0.2% |
| 医療診断（稀な疾患） | 陰性 99% / 陽性 1% |
| 製品不良品検出 | 良品 99% / 不良品 1% |

このような状況で「全部多数クラスと予測」すると正解率 99.8% になるが，**不正を一件も検知できていない**ので全く役に立たないモデルだ。

### 混同行列（Confusion Matrix）の理解

混同行列は分類モデルの結果を4つの分類で整理する：

```
                予測: Negative  予測: Positive
実際: Negative      TN               FP
実際: Positive      FN               TP
```

- **TP（True Positive）**: 陽性を陽性と正しく予測
- **TN（True Negative）**: 陰性を陰性と正しく予測
- **FP（False Positive）**: 陰性を陽性と誤予測（偽陽性）→ 「誤報」
- **FN（False Negative）**: 陽性を陰性と誤予測（偽陰性）→ 「見逃し」

### 各評価指標の意味

- **適合率（Precision）** = TP / (TP + FP)：「陽性と予測したうち実際に陽性の割合」→ **誤報を減らしたい**時に重視
- **再現率（Recall）** = TP / (TP + FN)：「実際の陽性のうち正しく予測した割合」→ **見逃しを減らしたい**時に重視（医療では特に重要）
- **F1スコア** = 2 × Precision × Recall / (Precision + Recall)：両者の調和平均

### ROC 曲線と AUC

ROC 曲線は**分類閾値を 0〜1 で変化させた時**の「偽陽性率（FPR）vs 真陽性率（TPR）」をプロットしたものだ。

- 完璧なモデル：左上の角に近い曲線
- ランダムな予測：対角線（AUC = 0.5）
- **AUC（Area Under the Curve）**：ROC 曲線の下の面積（0〜1）。1に近いほど良い

### PR 曲線が重要な場面

不均衡データでは ROC 曲線が「楽観的に見える」場合がある（TN が多いため FPR が小さくなりやすい）。**適合率-再現率曲線（PR曲線）** は少数クラスに注目した評価に向いている。

> **卒業研究での指針**: 不均衡データを扱う場合は，正解率だけでなく F1スコア，AUC，PR-AUC を報告することが学術的に求められる。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay,
    auc,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split


## 問題1　不均衡データの生成と確認　【骨格+選択】
***

### `make_classification` のパラメータ解説

`make_classification` は人工的な分類データを生成する。今回のパラメータ：
- `n_samples=2000`：2000件のデータ
- `n_features=20`：20個の特徴量
- `weights=[0.95, 0.05]`：**クラス0が95%，クラス1が5%** という不均衡設定
- `random_state=0`：再現性のため固定

### 不均衡度の確認が最初のステップ

データを受け取ったら，まず**クラスの分布を確認**することが重要だ。不均衡比率によって対処法が変わる：
- 5:1 程度：`class_weight="balanced"` で対処可能（今回）
- 100:1 以上：オーバーサンプリング（SMOTE）やアンダーサンプリングが必要

### 課題

下のコードセルは、不均衡データの生成とクラス分布の表示を用意してありますが、**核心（train/test 分割）はあなたが書きます**（`# ★あなたが書く★`）。まずは `minority_ratio = 0.05` のまま実行し、どれだけ偏ったデータかを確認してください。`minority_ratio`（★印の行）を変えれば不均衡の度合いを変えられます。

ここで生成したデータ（`X_train, X_test, y_train, y_test`）は問題2以降でそのまま使います。

そのうえで、次の **設計判断** に答えてください。

> **設計判断（不均衡への対処を選ぶ）**: このデータで「**陽性（クラス1＝不正/疾患など）の見逃し（FN）を絶対に減らしたい**」という状況だとします。次の対処から**1つ選び、理由**を解答用コードセルに書いてください。選んだ方針は問題2の実験につながります。
>
> - **(A) 何もしない**（そのまま `LogisticRegression` を学習）
> - **(B) `class_weight="balanced"`** … 少数クラスの誤分類のペナルティを大きくする
> - **(C) 決定閾値を下げる** … `predict_proba` のしきい値を 0.5 より小さくして陽性判定を増やす
> - **(D) オーバーサンプリング（SMOTE 等）** … 少数クラスを人工的に増やしてクラス比を是正
> - **(E) アンダーサンプリング** … 多数クラスを間引いてクラス比を是正
> - **(F) 決定閾値を上げる（0.5→0.7）** … 陽性判定を厳しくする
>
> ヒント：「見逃しを減らしたい＝Recall を上げたい」とき、どの対処が直接効くでしょうか。逆効果になる選択肢も混ざっています（例えば (F) は見逃しを増やす方向です）。**やみくもに全部やる**のではなく、**この状況に合うもの**を選び、なぜ効くと思うかを書いてください（複数挙げてもよいが主役を1つ）。


In [ ]:
# === 完成済みコード：実行して不均衡の度合いを確認してください ===
# === ★ここを変えて実験する★：少数クラス（クラス1）の比率（まずは 0.05 のまま実行）===
minority_ratio = 0.05

X, y = make_classification(
    n_samples=2000,
    n_features=20,
    weights=[1 - minority_ratio, minority_ratio],
    random_state=0,
)
# ★あなたが書く★：X, y を train/test に 8:2 で分割する（1行。stratify=y で不均衡比率を保つ）
#   ヒント: train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_train, X_test, y_train, y_test = ___

counts = pd.Series(y_train).value_counts().sort_index()
ratio = (counts / counts.sum() * 100).round(1)
dist = pd.concat([counts.rename("件数"), ratio.rename("比率%")], axis=1)
dist.index.name = "クラス"
print("訓練データのクラス分布:")
print(dist)


In [ ]:
# === ✍️ 問題1 解答（採点対象）===

# (1-a) 観察：訓練データのクラス0／クラス1の件数と比率
observation_1_a = """
"""

# (1-b) 設計判断：見逃し（FN）を減らしたい状況で選んだ対処（A〜F）：(　)
# (A) 何もしない（class_weight なし・閾値 0.5）
# (B) class_weight='balanced'
# (C) 閾値を下げる（Recall 重視）
# (D) SMOTE などで少数クラスを増やす
# (E) 多数クラスを減らす（アンダーサンプリング）
# (F) 別のモデル（例: 木系）に変える
design1_choice = ""

# (1-c) その理由（なぜこの状況に合うか／なぜ見逃しを減らせると思うか）
answer_1_c = """
"""



## 問題2　class_weight と決定閾値の実験　【実験】
***

### `class_weight="balanced"` の仕組み

`class_weight="balanced"` を指定すると，scikit-learn はクラスの出現頻度に反比例した重みを自動計算する：

```
クラスiの重み = 全サンプル数 / (クラス数 × クラスiのサンプル数)

例: クラス0が1900件，クラス1が100件，計2000件の場合
  クラス0の重み = 2000 / (2 × 1900) ≈ 0.53
  クラス1の重み = 2000 / (2 × 100) = 10.0
```

少数クラス（クラス1）の誤分類に大きなペナルティをかけることで，モデルが少数クラスを積極的に検出するようになる。

### Precision と Recall のトレードオフを理解する

`class_weight="balanced"` を使うと：
- 再現率（クラス1の見逃し率が下がる）↑
- 適合率（クラス1と予測したうちの正解率）↓ （になることが多い）

このトレードオフは**どちらのエラーがより重大か**によって判断する：
- 癌の診断：見逃し（FN）が致命的 → 再現率を優先
- スパムフィルター：誤判定（FP）が嫌 → 適合率を優先

### 課題

下のコードセルは `evaluate(...)` の骨組みを用意してありますが、**核心（予測確率を閾値で 0/1 に変換する1行）はあなたが書きます**（`# ★あなたが書く★`）。問題1で選んだ対処（class_weight や決定閾値）が、実際に**見逃し（FN）を減らせるか**を実験で確かめます。

`settings`（★印のリスト）の組合せを **最低5通り**、しかも **class_weight と threshold の両方の軸**を動かして試し、結果を **✍️ 解答用コードセルの実験ログ**に記録してください。

| class_weight | threshold（決定閾値） |
|---|---|
| `None`, `"balanced"` | `0.5`（標準）, `0.3`, `0.2` など |

そのうえで考察してください：

> **考察1**: `class_weight=None, threshold=0.5`（何もしない）のとき、クラス1の **FN（見逃し）** は何件でしたか？ そこから `class_weight="balanced"` や閾値を下げると、FN と Precision はそれぞれどう動きましたか？
>
> **考察2**: Recall を上げると Precision が下がる「**トレードオフ**」が観察できるはずです。なぜ「閾値を下げる／少数クラスの重みを上げる」と Recall は上がるのに Precision は下がるのか、混同行列の TP・FP・FN の動きに触れて説明してください。
>
> **考察3**: 問題1で選んだ対処は、実験でも狙いどおり見逃しを減らせましたか？ 想定と違った場合、なぜだと思いますか？


In [ ]:
# === 完成済みコード：class_weight と決定閾値を変えて評価を観察してください ===
def evaluate(class_weight, threshold):
    model = LogisticRegression(max_iter=1000, class_weight=class_weight)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]   # クラス1（陽性）の予測確率
    # ★あなたが書く★：予測確率 proba を閾値 threshold で 0/1 に変換して y_pred を作る（1行）
    #   ヒント: (proba >= threshold) は True/False の配列。.astype(int) で 0/1 に
    y_pred = ___
    cm = confusion_matrix(y_test, y_pred)
    fn = cm[1, 0]                                # 見逃し（陽性を陰性と誤判定 = FN）
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    return prec, rec, f1, fn

# === ★ここを変えて実験する★：(class_weight, threshold) の組合せ ===
settings = [
    (None,       0.5),   # 何もしない（標準）
    ("balanced", 0.5),   # class_weight だけ変える
    (None,       0.3),   # 決定閾値だけ下げる
    (None,       0.2),   # さらに閾値を下げる
    ("balanced", 0.3),   # 両方変える
    # 例: ("balanced", 0.2) などを追加してよい
]

print(f"{'class_weight':12s} {'threshold':>9s} | {'Precision':>9s} {'Recall':>7s} {'F1':>6s} {'FN(見逃し)':>9s}")
print("-" * 64)
for cw, th in settings:
    prec, rec, f1, fn = evaluate(cw, th)
    cw_name = "balanced" if cw == "balanced" else "none"
    print(f"{cw_name:12s} {th:9.2f} | {prec:9.3f} {rec:7.3f} {f1:6.3f} {fn:9d}")


In [ ]:
# === ✍️ 問題2 解答（採点対象）===
import pandas as pd


# (2-a) 実験ログ（5通り以上・class_weight と threshold の両軸を動かす）
experiment_log = pd.DataFrame([
    {'row': 1, 'class_weight': 'none', 'threshold': 0.5, 'precision': None, 'recall': None, 'f1': None, 'fn': None},
    {'row': 2, 'class_weight': 'balanced', 'threshold': 0.5, 'precision': None, 'recall': None, 'f1': None, 'fn': None},
    {'row': 3, 'class_weight': 'none', 'threshold': 0.3, 'precision': None, 'recall': None, 'f1': None, 'fn': None},
    {'row': 4, 'class_weight': 'none', 'threshold': 0.2, 'precision': None, 'recall': None, 'f1': None, 'fn': None},
    {'row': 5, 'class_weight': 'balanced', 'threshold': 0.3, 'precision': None, 'recall': None, 'f1': None, 'fn': None},
    {'row': 6, 'class_weight': None, 'threshold': None, 'precision': None, 'recall': None, 'f1': None, 'fn': None},
])

# (2-b) 何もしない（none, 0.5）のときのクラス1の FN（見逃し）は何件か
answer_2_b = """
"""

# (2-c) 考察1：class_weight や閾値を変えると FN と Precision はそれぞれどう動いたか
reflection1 = """
"""

# (2-d) 考察2：Recall↑ と Precision↓ のトレードオフが起きる理由（TP・FP・FN の動きに触れて）
reflection2 = """
"""

# (2-e) 考察3：問題1で選んだ対処は狙いどおり見逃しを減らせたか／違ったら理由
reflection3 = """
"""



## 問題3　ROC 曲線と AUC　【説明】
***

### ROC 曲線の読み方

ROC 曲線は「閾値を変化させたとき，どれだけ上手く分類できるか」を可視化する：

```
縦軸：真陽性率（TPR = 再現率）= TP / (TP + FN)
横軸：偽陽性率（FPR）= FP / (FP + TN)

・左上の角（TPR=1, FPR=0）が理想
・対角線（y=x）はランダムな予測（AUC=0.5）
```

**AUC の解釈**:
- ランダムに選んだ陽性サンプルのスコアが，ランダムに選んだ陰性サンプルのスコアより高い確率 = AUC

つまり AUC=0.9 なら，「陽性と陰性をランダムに1件ずつ選んだとき，モデルが陽性を高くスコアリングする確率が90%」という意味だ。

### class_weight なし vs あり の AUC 比較

今回は `class_weight="balanced"` のモデルの ROC 曲線を描く。**AUC はクラスの不均衡に比較的ロバスト**（頑健）なため，正解率が全く役に立たない不均衡データでも，モデルの性能を適切に評価できる。

### 課題

下のコードセルは、`class_weight="balanced"` のモデルの **ROC 曲線**を描き、**AUC** を出すところまで **完成済み**です。今回はコードを書くのではなく、**ROC 曲線が何を表しているのかを読み解く**のが目的です。

各行の `# 説明:` の右に、その行が何をしているかを**自分の言葉で**書いてください（コードは変更しない）。書き終えたらセルを実行し、曲線と AUC が出ることを確認してください。

説明を書くときは、次の問いを意識してください：

- `predict_proba(...)[:, 1]` の `[:, 1]` はなぜ必要か？（何の確率を取り出しているか）
- ROC 曲線の**横軸 FPR**・**縦軸 TPR** はそれぞれ混同行列のどの値から計算されるか？（FPR = FP/(FP+TN)、TPR = TP/(TP+FN)）
- 対角線（点線）は何を表し、AUC=0.5 とはどういう状態か？
- AUC が「ランダムに選んだ陽性のスコアが、ランダムに選んだ陰性のスコアより高い確率」と等しいのはなぜ、直感的にどういう意味か？

> **考察**: ROC 曲線は閾値を 0〜1 まで動かしたときの (FPR, TPR) の軌跡です。問題2で「決定閾値を下げる」と Recall が上がったことと、ROC 曲線上を**どの向きに動くこと**が対応していますか？（TPR=Recall であることに注目）

In [ ]:
# 各行の # 説明: に自分の言葉で意味を書いてください（コードは変更しない）
# 説明は AI に書かせず、自分で書くこと

model = LogisticRegression(max_iter=1000, class_weight="balanced")  # 説明:
model.fit(X_train, y_train)                                         # 説明:
proba = model.predict_proba(X_test)[:, 1]                           # 説明:（なぜ [:, 1] ？）

fpr, tpr, thresholds = roc_curve(y_test, proba)                     # 説明:（fpr・tpr とは何か）
roc_auc = auc(fpr, tpr)                                             # 説明:（AUC は何の面積か）

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC (AUC={roc_auc:.3f})")               # 説明:（横軸=fpr, 縦軸=tpr の意味）
plt.plot([0, 1], [0, 1], "k--", label="ランダム（AUC=0.5）")       # 説明:（対角線は何を表す？）
plt.xlabel("偽陽性率 FPR = FP / (FP + TN)")
plt.ylabel("真陽性率 TPR = TP / (TP + FN)（=再現率）")
plt.title("ROC 曲線（class_weight=balanced）")
plt.legend()
plt.show()
print(f"AUC = {roc_auc:.4f}")


In [ ]:
# === ✍️ 問題3 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (3-a) `predict_proba(...)[:, 1]` の `[:, 1]` で何を取り出しているか
answer_3_a = """
"""

# (3-b) ROC の横軸 FPR・縦軸 TPR は混同行列のどの値から計算されるか
answer_3_b = """
"""

# (3-c) 対角線（点線）は何を表し、AUC=0.5 とはどういう状態か
answer_3_c = """
"""

# (3-d) 考察：閾値を下げて Recall を上げることは、ROC 曲線上をどの向きに動くことに対応するか
answer_3_d = """
"""



## 問題4　適合率-再現率曲線（PR曲線）　【説明+選択】
***

### なぜ不均衡データでは PR 曲線が重要か

不均衡データ（クラス0が95%）では ROC 曲線が「楽観的に見える」問題がある：

```
TN が非常に多い
→ FPR = FP / (FP + TN) が自動的に小さくなりやすい
→ ROC 曲線が左上に寄る
→ AUC が実際より高く見える
```

PR 曲線は TN を使わないため，少数クラス（クラス1）の検出性能をよりシビアに評価できる。

### PR 曲線の読み方

```
縦軸：適合率（Precision）= TP / (TP + FP)
横軸：再現率（Recall）  = TP / (TP + FN)

・右上（Precision=1, Recall=1）が理想
・曲線の下の面積（PR-AUC または Average Precision）が高いほど良い
・不均衡データでは PR-AUC のベースラインは「少数クラスの比率」（今回は0.05）
```

### 課題

下のコードセルは、問題3のモデルについて **PR 曲線**を描き、**PR-AUC（Average Precision）** とランダム基準（少数クラスの比率）を出すところまで **完成済み**です。問題3の `model` と `proba` をそのまま再利用します。

**(1) 説明**：各行の `# 説明:` の右に、その行が何をしているかを**自分の言葉で**書いてください（コードは変更しない）。特に、PR 曲線の**横軸 Recall・縦軸 Precision** がそれぞれ混同行列のどの値から計算されるか、ランダム基準がなぜ「少数クラスの比率」になるのかを意識してください。

**(2) 選択**：この**不均衡データ（クラス1が約5%）**でモデルを評価・報告するとき、**主指標として何を報告すべき**でしょうか。次から1つ選び、**理由**を解答用コードセルに書いてください。

> - **(A) Accuracy（正解率）**
> - **(B) ROC-AUC**
> - **(C) PR-AUC（Average Precision）**
> - **(D) 閾値0.5での F1 スコア**
> - **(E) 閾値0.5での Precision**
> - **(F) 「全部多数クラス」と予測したときの正解率との比較だけ**
>
> ヒント：不均衡データでは **TN が非常に多い** ため、`FPR = FP/(FP+TN)` が小さくなりやすく ROC 曲線は左上に寄って「楽観的」に見えます。一方 PR 曲線は **TN を使わない**ので少数クラスの検出をシビアに評価します。(A)(F) のように不均衡で誤解を招く指標も混ざっています。**なぜそれを主指標にするのか**、TN の多さに触れて説明してください。

> **考察**: 出力された **ROC の AUC** と **PR-AUC** の値を比べてください。どちらが高い値になりましたか？ その差は何を意味しますか？（同じモデルなのに2つの指標で値が違う理由）

In [ ]:
# 各行の # 説明: に自分の言葉で意味を書いてください（コードは変更しない）
# 問題3の model・proba を再利用します

precision, recall, thresholds = precision_recall_curve(y_test, proba)  # 説明:（返り値3つの意味）
pr_auc = average_precision_score(y_test, proba)                        # 説明:（PR-AUC とは何か）
baseline = (y_test == 1).mean()                                        # 説明:（なぜ少数クラスの比率？）

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f"PR 曲線 (PR-AUC={pr_auc:.3f})")    # 説明:（横軸=recall, 縦軸=precision）
plt.axhline(baseline, color="red", linestyle="--",
            label=f"ランダム基準 ({baseline:.3f})")                    # 説明:（この水平線の意味）
plt.xlabel("再現率 Recall = TP / (TP + FN)")
plt.ylabel("適合率 Precision = TP / (TP + FP)")
plt.title("適合率-再現率（PR）曲線")
plt.legend()
plt.show()
print(f"PR-AUC (Average Precision) = {pr_auc:.4f}")
print(f"参考：ROC の AUC = {roc_auc:.4f}")


In [ ]:
# === ✍️ 問題4 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (4-a) PR 曲線のランダム基準がなぜ「少数クラスの比率」になるのか
answer_4_a = """
"""

# (4-b) 選択：不均衡データで報告すべき主指標（A〜F）：(　)
# (A) Accuracy
# (B) Precision
# (C) Recall
# (D) F1
# (E) ROC-AUC
# (F) PR-AUC（Average Precision）
answer_4_b = ""

# (4-c) その理由（なぜ不均衡では PR の方が厳しい評価になるか。TN の多さに触れて）
answer_4_c = """
"""

# (4-d) 考察：出力された ROC の AUC と PR-AUC の値の比較／差が意味すること
answer_4_d = """
"""
# ROC の AUC：(　)
roc_auc = ""
# PR-AUC：(　)
pr_auc = ""
# 差が出る理由
差が出る理由 = ""

